# 01 Baseline CoM and Augmentation

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`HGB_Normal_Combined_v2.ipynb`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'e5f86b39e01a7b4a92727b0052ca3a6d03f15adf'}}

## HGB_Normal_Combined_v2

원본: [05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb)  
Git blob: `e5f86b39e01a7b4a92727b0052ca3a6d03f15adf`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

원본 Notebook의 Markdown 및 실행 코드 셀을 순서대로 보존한다. 기존 출력·이미지는 복제하지 않는다. 원본 Colab 환경/실행 순서는 원본 링크 참조.

### 원본 셀 0

# HGB Normal + Combined 최적화

**조장님 GitHub의 데이터 로드 코드와 전처리 함수를 그대로 불러와서** 모델만 HGB로 바꾼 노트북입니다. 그래서 분할·손글씨·CoM·크기/획 증강이 MLP 실험과 완전히 같아요.

**사용법: 고칠 곳 없이 위에서부터 순서대로 실행만 하면 됩니다.** 결과는 내 구글 드라이브 `HGB_results/` 폴더에 저장돼요.

| 셀 | 내용 | 보고서 절 |
|---|---|---|
| N1 | 원본 HGB (기준) | 기준 모델 |
| N2 | CoM 정렬 + 교차평가 | 중심 정렬 |
| N3 | CoM + 크기 30%·획 28% 증강 | 균형 증강 |
| N4 | 하이퍼파라미터 3개를 하나씩 | 하이퍼파라미터 |
| C1 | Combined 작성자 보류 4-fold OOF | Combined |

### 원본 셀 1

## 0. 저장소 가져오기 + 드라이브 연결

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell2
!test -d /content/Machine_Learning || git clone --depth 1 https://github.com/ryu980920/Machine_Learning.git /content/Machine_Learning

from google.colab import drive
drive.mount('/content/drive')
import os; OUT = '/content/drive/MyDrive/HGB_results/'; os.makedirs(OUT, exist_ok=True)

import sys, time, numpy as np, pandas as pd
from pathlib import Path
REPO = Path('/content/Machine_Learning')
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Combined/scripts'))
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Normal/scripts'))

# 조장님 코드 그대로 사용
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process          # CoM 정렬
import exp_cmb005_balanced_scale_stroke as L             # prepareraw: 크기·획 증강 + CoM

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

### 원본 셀 3

## 1. 데이터 로드 (공식 Train 76,800 / Validation 19,200, 손글씨 4명 × 640장)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell4
off = load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz')
wr  = load_writers(REPO/'02_data/Hand_made')
WRITERS = list(L.WRITERS); LABELS = '0123456789ABCDEF'
ytr, yva = off['y_train'], off['y_val']
yhm = np.concatenate([wr[w]['y'] for w in WRITERS])
whm = np.concatenate([[w]*len(wr[w]['y']) for w in WRITERS])
print(off['X_train'].shape, off['X_val'].shape, len(yhm))   # (76800,784) (19200,784) 2560

### 원본 셀 5

## 2. 공통 함수 (평가·학습)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell6
PAIRS = [('D','0'), ('0','D'), ('B','8'), ('8','B'), ('5','F'), ('7','9')]
results = []

def hm_scores(pred, ytrue, writers):
    isd = ytrue < 10
    r = dict(hm_acc=accuracy_score(ytrue, pred), hm_f1=f1_score(ytrue, pred, average='macro'),
             hm_digit=accuracy_score(ytrue[isd], pred[isd]), hm_letter=accuracy_score(ytrue[~isd], pred[~isd]))
    for w in WRITERS: r[f'w{w}'] = accuracy_score(ytrue[writers==w], pred[writers==w])
    for a, b in PAIRS: r[f'{a}->{b}'] = int(np.sum((ytrue==LABELS.index(a)) & (pred==LABELS.index(b))))
    return r

def evaluate(name, model, Xva, Xhm, extra=None):
    pv = model.predict(Xva); ph = model.predict(Xhm)
    row = dict(name=name, val_acc=accuracy_score(yva, pv), val_f1=f1_score(yva, pv, average='macro'),
               **hm_scores(ph, yhm, whm), **(extra or {}))
    results.append(row); pd.DataFrame(results).to_csv(OUT+'HGB_results.csv', index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})
    return ph

def hgb(**kw):
    p = dict(max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=42)
    p.update(kw); return HistGradientBoostingClassifier(**p)

def fit(m, X, y):
    t = time.time(); m.fit(X, y); return dict(fit_s=round(time.time()-t), iters=m.n_iter_)

### 원본 셀 7

## N1. 원본 (전처리 없음, 기준 모델)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell8
Xhm_raw = np.vstack([wr[w]['X'] for w in WRITERS])
m1 = hgb(); info = fit(m1, off['X_train'], ytr)
ph1 = evaluate('N1_raw', m1, off['X_val'], Xhm_raw, info)

### 원본 셀 9

## N2. CoM 정렬 (학습·검증·손글씨 모두 같은 처리)
+ 교차평가: 원본 모델에 CoM 입력만 넣기 → '이동 효과'와 '재학습 효과' 분리

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell10
Xtr_com, tw_com, _, _ = L.prepareraw(off, wr, 'com_only')
Xva_com = process(off['X_val'], 'com', 20)
Xhm_com = np.vstack([tw_com[w] for w in WRITERS])

m2 = hgb(); info = fit(m2, Xtr_com, ytr)
evaluate('N2_CoM', m2, Xva_com, Xhm_com, info)
evaluate('N1모델 + CoM입력 (교차)', m1, Xva_com, Xhm_com)

### 원본 셀 11

## N3. CoM + 크기 30%·획 28% 증강 (조장님 Combined 최종 전처리)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell12
Xtr_aug, tw_aug, info_aug, _ = L.prepareraw(off, wr, 'scale30_stroke28')
print(info_aug)
m3 = hgb(); info = fit(m3, Xtr_aug, ytr)
evaluate('N3_CoM+size30_stroke28', m3, Xva_com, Xhm_com, info)
pd.DataFrame(results)

### 원본 셀 13

## N4. 하이퍼파라미터 — 하나씩 바꾸고, 공식 Validation이 가장 높은 값을 고정한 뒤 다음으로
아래 `USE_AUG`는 N2/N3 중 손글씨 정확도가 더 좋았던 쪽으로 두세요.

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell14
USE_AUG = True                    # N3가 N2보다 좋으면 True, 아니면 False
Xtr_best = Xtr_aug if USE_AUG else Xtr_com

best = {}
for pname, values in [('learning_rate', [0.05, 0.1, 0.2]),
                      ('max_leaf_nodes', [15, 31, 63]),
                      ('l2_regularization', [0.0, 1.0, 10.0])]:
    sc = {}
    for v in values:
        kw = {**best, pname: v}
        m = hgb(**kw); info = fit(m, Xtr_best, ytr)
        evaluate(f'N4_{pname}={v}', m, Xva_com, Xhm_com, {**info, **kw})
        sc[v] = results[-1]['val_acc']
    best[pname] = max(sc, key=sc.get); print('>>> 선택', pname, '=', best[pname])
print('Normal 최종 설정:', best)

### 원본 셀 15

## C1. Combined — 작성자 1명 보류 · 나머지 3명 1,920장을 학습에 추가 · 4번 반복 → OOF

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell16
def combined_oof(Xtr, hm_map, kw, name):
    oof = np.zeros(len(yhm), dtype=int)
    for w in WRITERS:
        others = [o for o in WRITERS if o != w]
        X = np.vstack([Xtr] + [hm_map[o] for o in others])
        y = np.concatenate([ytr] + [wr[o]['y'] for o in others])
        m = hgb(**kw); fit(m, X, y)
        oof[whm == w] = m.predict(hm_map[w])
        print(f'  보류 {w}: {accuracy_score(yhm[whm==w], oof[whm==w]):.4f}')
    row = dict(name=name, **hm_scores(oof, yhm, whm), **kw)
    results.append(row); pd.DataFrame(results).to_csv(OUT+'HGB_results.csv', index=False)
    print(row); return oof

raw_map = {w: wr[w]['X'] for w in WRITERS}
oof_raw  = combined_oof(off['X_train'], raw_map, {}, 'C1_raw (전처리·튜닝 없음)')
oof_best = combined_oof(Xtr_best, tw_com, best, 'C1_best (Normal 최종 전처리·설정)')

### 원본 셀 17

## 혼동행렬 그림 + 최종 표

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_Normal_Combined_v2.ipynb#cell18
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(2, 2, figsize=(16, 15))
for a, (p, t) in zip(ax.ravel(), [(ph1,'N1 raw (Normal)'), (m3.predict(Xhm_com),'N3 aug (Normal)'),
                                  (oof_raw,'C1 raw (Combined OOF)'), (oof_best,'C1 best (Combined OOF)')]):
    ConfusionMatrixDisplay.from_predictions(yhm, p, labels=range(16), display_labels=list(LABELS),
                                            normalize='true', values_format='.2f', ax=a, colorbar=False)
    a.set_title(t)
plt.tight_layout(); plt.savefig(OUT+'HGB_confusion.png', dpi=130); plt.show()
pd.DataFrame(results)

## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")